In [8]:
import pandas as pd # Import the pandas library for data manipulation and analysis.
import numpy as np # Import the numpy library for numerical operations, especially array manipulations.
from sklearn.model_selection import train_test_split, RandomizedSearchCV # Import train_test_split for dividing data into training/testing sets, and RandomizedSearchCV for hyperparameter tuning.
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Import StandardScaler for feature scaling (numerical) and OneHotEncoder for converting categorical features into a numerical format.
from sklearn.compose import ColumnTransformer # Import ColumnTransformer to apply different transformations to different columns of the data.
from sklearn.pipeline import Pipeline # Import Pipeline to chain together multiple data processing steps and the estimator.
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix # Import various metrics for evaluating classification model performance.
from xgboost import XGBClassifier # Import XGBClassifier, a powerful gradient boosting machine learning algorithm for classification tasks.

# =====================================================================
# 1. Load Data and Prepare Features
# =====================================================================
df = pd.read_csv('/content/train.csv') # Load the training dataset from the specified CSV file path into a pandas DataFrame named 'df'.

# Separate features (X) from target variable (y) and drop non-influential columns
X = df.drop(columns=['id', 'CustomerId', 'Surname', 'Exited']) # Create a feature DataFrame 'X' by dropping 'id', 'CustomerId', 'Surname' (non-predictive unique identifiers) and 'Exited' (the target variable) from 'df'.
y = df['Exited'] # Create the target variable Series 'y' containing the 'Exited' column from 'df'.

# =====================================================================
# 2. Split Data (to avoid Data Leakage)
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Split the feature and target data into training (80%) and testing (20%) sets. 'random_state' ensures reproducibility, and 'stratify=y' ensures that the proportion of target classes is the same in both training and testing sets.

# =====================================================================
# 3. Build Data Preprocessing Pipeline
# =====================================================================
numeric_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary'] # Define a list of numerical feature column names.
categorical_features = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember'] # Define a list of categorical feature column names.

preprocessor = ColumnTransformer( # Create a ColumnTransformer to apply different preprocessing steps to different columns.
    transformers=[ # Define the list of transformers to apply.
        ('num', StandardScaler(), numeric_features), # Apply StandardScaler (scales features to have zero mean and unit variance) to the 'numeric_features'.
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features) # Apply OneHotEncoder (converts categorical features into a one-hot numerical array) to the 'categorical_features'. 'handle_unknown='ignore'' prevents errors if an unseen category appears, and 'drop='first'' avoids multicollinearity.
    ])

# =====================================================================
# 4. Handle Data Imbalance and Prepare the Model (XGBoost)
# =====================================================================
ratio = float(np.sum(y_train == 0) / np.sum(y_train == 1)) # Calculate the ratio of the majority class (0) to the minority class (1) in the training target variable. This is used to handle class imbalance in XGBoost.

xgb_model = XGBClassifier(scale_pos_weight=ratio, random_state=42, eval_metric='logloss') # Initialize an XGBoost Classifier model. 'scale_pos_weight' uses the calculated ratio to give more weight to the minority class, 'random_state' ensures reproducibility, and 'eval_metric='logloss'' sets the evaluation metric for early stopping or monitoring.

full_pipeline = Pipeline(steps=[ # Create a scikit-learn Pipeline to sequentially apply the preprocessor and then the XGBoost classifier.
    ('preprocessor', preprocessor), # The first step in the pipeline is the 'preprocessor' defined earlier.
    ('classifier', xgb_model) # The second step is the 'xgb_model' classifier.
])

# =====================================================================
# 5. Set up Hyperparameter Tuning
# =====================================================================
param_distributions = { # Define a dictionary of hyperparameter distributions for RandomizedSearchCV to search through.
    'classifier__n_estimators': [100, 200, 300, 500], # Number of boosting rounds (trees) for the XGBoost classifier.
    'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2], # Step size shrinkage used in updating to prevent overfitting.
    'classifier__max_depth': [3, 5, 7, 9], # Maximum depth of a tree. Increasing this value will make the model more complex and more likely to overfit.
    'classifier__subsample': [0.6, 0.8, 1.0], # Subsample ratio of the training instance. Setting it to 0.5 means that XGBoost would randomly sample half of the training data prior to growing trees.
    'classifier__colsample_bytree': [0.6, 0.8, 1.0] # Subsample ratio of columns when constructing each tree.
}

In [9]:
# =====================================================================
# 6. Perform Randomized Search for Hyperparameter Tuning
# =====================================================================
random_search = RandomizedSearchCV(
    estimator=full_pipeline, # The pipeline containing preprocessor and classifier.
    param_distributions=param_distributions, # The dictionary of hyperparameter distributions to sample from.
    n_iter=50, # Number of parameter settings that are sampled. A higher number means a more thorough search but takes longer.
    scoring='roc_auc', # The metric used to evaluate the model performance. 'roc_auc' is appropriate for imbalanced classification.
    cv=5, # Number of cross-validation folds. Using 5-fold cross-validation.
    verbose=1, # Controls the verbosity: higher values mean more messages.
    random_state=42, # Ensures reproducibility of the random sampling.
    n_jobs=-1 # Uses all available CPU cores for computation to speed up the search.
)

# Fit RandomizedSearchCV to the training data. This will perform the search.
random_search.fit(X_train, y_train)

# Get the best estimator (model) found by the random search.
best_model = random_search.best_estimator_

# Print the best hyperparameters found.
print("\nBest Hyperparameters found by RandomizedSearchCV:")
print(random_search.best_params_)

# Print the best ROC AUC score achieved with the best hyperparameters.
print(f"\nBest ROC AUC Score (from cross-validation): {random_search.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Hyperparameters found by RandomizedSearchCV:
{'classifier__subsample': 0.8, 'classifier__n_estimators': 200, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 1.0}

Best ROC AUC Score (from cross-validation): 0.9352


In [10]:
# Fit the full pipeline on the training data
full_pipeline.fit(X_train, y_train)

# Make predictions on the training set
y_train_pred = full_pipeline.predict(X_train)
y_train_proba = full_pipeline.predict_proba(X_train)[:, 1]

# Make predictions on the test set
y_test_pred = full_pipeline.predict(X_test)
y_test_proba = full_pipeline.predict_proba(X_test)[:, 1]

# Print classification report for training set
print("\n--- Classification Report (Training Set) ---")
print(classification_report(y_train, y_train_pred))

# Print classification report for test set
print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_test_pred))

# Print ROC AUC Score for training set
print(f"ROC AUC Score (Training Set): {roc_auc_score(y_train, y_train_proba):.4f}")

# Print ROC AUC Score for test set
print(f"ROC AUC Score (Test Set): {roc_auc_score(y_test, y_test_proba):.4f}")


--- Classification Report (Training Set) ---
              precision    recall  f1-score   support

         0.0       0.99      0.96      0.98      9598
         1.0       0.85      0.98      0.91      2402

    accuracy                           0.96     12000
   macro avg       0.92      0.97      0.94     12000
weighted avg       0.97      0.96      0.96     12000


--- Classification Report (Test Set) ---
              precision    recall  f1-score   support

         0.0       0.94      0.92      0.93      2400
         1.0       0.69      0.76      0.73       600

    accuracy                           0.89      3000
   macro avg       0.82      0.84      0.83      3000
weighted avg       0.89      0.89      0.89      3000

ROC AUC Score (Training Set): 0.9951
ROC AUC Score (Test Set): 0.9195


In [12]:
# Load the test data
test_df = pd.read_csv('/content/test.csv')

# Keep track of the 'id' column for the submission file
test_ids = test_df['id']

# Prepare the test features (drop 'id', 'CustomerId', 'Surname')
X_test_submission = test_df.drop(columns=['id', 'CustomerId', 'Surname'])

# Predict probabilities on the test set using the fitted pipeline
# The full_pipeline already includes preprocessing steps
submission_probabilities = full_pipeline.predict_proba(X_test_submission)[:, 1]

# Load the sample submission file to understand the required format
sample_submission_df = pd.read_csv('/content/sample_submission.csv')

# Create the submission DataFrame
submission_df = pd.DataFrame({'id': test_ids, 'Exited': submission_probabilities})

# Ensure the 'Exited' column has the correct data type if needed (e.g., float)
submission_df['Exited'] = submission_df['Exited'].astype(float)

# Save the submission file
submission_file_path = 'submission.csv'
submission_df.to_csv(submission_file_path, index=False)

print(f"Submission file created successfully at: {submission_file_path}")
print("First 5 rows of the submission file:")
print(submission_df.head())

Submission file created successfully at: submission.csv
First 5 rows of the submission file:
      id    Exited
0  15000  0.027974
1  15001  0.030181
2  15002  0.104784
3  15003  0.001116
4  15004  0.962078


In [13]:
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')